# Evaluation notebook: where did a case go right and wrong?

Pick a `CASE` and a `RUN`, then work top to bottom:

1. **Scorecard**: decision match, official reasons P/R/F1, and conditions P/R/F1 (our extension, same judge).
2. **Alignment**: reference vs generated side by side, coloured by judge similarity, plus missing/extra.
3. **Failure triage**: for each miss or extra, the pipeline evidence that explains it, and a place to record your diagnosis.
4. **Intermediate data**: facts, constraints, routing, per-issue assessments, the policy the assessor actually saw.
5. **Trace**: the exact prompt and reply of every LLM call.
6. **Roll-up** over the eval cases, and **A/B** comparison of two runs.

Budget safety: the judge is cached on disk, so re-scoring is free; the judge refuses to run on anything but the cheap model unless you flip `ALLOW_EXPENSIVE_JUDGE`; the optional re-run cell is off by default and cheap-only by default.

Only eval cases (001-005) have a reference decision. Test cases (006-010) still get the intermediate-data and trace views.

Make a run snapshot (free if the LLM cache is warm):
`uv run assess case-001 case-002 case-003 case-004 case-005 --run baseline --cache-only`

In [ ]:
import json
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd
from IPython.display import HTML, Markdown, display

from config import load_settings
from decision import Decision
from evaluator.diagnostics import (
    DEV_CASES, CachedJudge, Label, ab_table, alignment_rows, baseline_name, compare_kind,
    label_summary, link_evidence, link_facts, list_runs, load_labels, load_meta,
    load_reference, load_run, model_mismatch, policy_seen, rerun_cases, save_label,
    score_case, score_run, scorecard, set_baseline, settings_for_models, trace_frame,
)
from evaluator.evaluator import PlanningDecisionSimpleEvaluator
from llm import LLMClient

# ---- choose what to look at -------------------------------------------------
CASE = "case-001"          # eval: case-001..005 (scored); test: 006..010 (no score)
RUN = "@baseline"          # "@baseline" = current baseline pointer, a run name, or None for the latest copy
COMPARE_RUN = None         # a second run name for the A/B section
SAMPLES = 1                # >1 re-judges independently to show judge noise (costs judge calls)
ALLOW_EXPENSIVE_JUDGE = False
# -----------------------------------------------------------------------------

settings = load_settings()
settings = settings.all_cheap()
client = LLMClient(settings)
if settings.judge_model != settings.cheap_model and not ALLOW_EXPENSIVE_JUDGE:
    raise SystemExit(
        f"Judge model {settings.judge_model} is not the cheap model; "
        "set ALLOW_EXPENSIVE_JUDGE = True to proceed."
    )
judge = CachedJudge(
    PlanningDecisionSimpleEvaluator(settings.judge_model, client.raw),
    settings.outputs_dir / "eval_cache",
)
BASELINE = baseline_name(settings)   # the run new methods are compared against
if RUN == "@baseline":
    RUN = BASELINE

STATUS_COLOURS = {"matched": "#d4edda", "weak": "#fff3cd", "missing": "#f8d7da", "extra": "#d6e4f5"}


def show(frame, status_col="status"):
    """Render a table with wrapped text, coloured by status when present."""
    styler = frame.style.hide(axis="index").set_properties(
        **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
    )
    if status_col in frame:
        styler = styler.apply(
            lambda row: [f"background-color: {STATUS_COLOURS.get(row[status_col], '')}"] * len(row),
            axis=1,
        )
    display(styler)


print("runs available:", list_runs(settings) or "none yet (see command above)")
print("current baseline:", BASELINE, "| viewing run:", RUN)

## 1. Load the run and the reference

In [ ]:
run_case = load_run(settings, CASE, RUN)
reference = load_reference(settings, CASE)
print(f"{CASE} / run={RUN}: predicted {run_case.decision.decision.value}, "
      f"{len(run_case.decision.reasons)} reasons, {len(run_case.decision.conditions)} conditions, "
      f"{len(run_case.trace)} traced LLM calls")
if reference is None:
    print("Test case: no reference decision, so no scores. The intermediate-data and trace sections still work.")
else:
    print(f"Reference: {reference.decision.value}, {len(reference.reasons)} reasons, "
          f"{len(reference.conditions)} conditions")
if not run_case.trace:
    print("No trace.jsonl in this run: re-create it with `uv run assess ... --run <name>` "
          "(add --cache-only for a free replay).")

## 2. Scorecard

Official metric = reasons only (same maths as `evaluator.scores`). Conditions use the same judge and formula but are **our extension**; they are `NaN` when neither side has any (e.g. a refusal). With `SAMPLES > 1` the range shows judge noise.

In [ ]:
score = await score_case(judge, CASE, reference, run_case.decision, SAMPLES) if reference else None
if score:
    display(scorecard([score]).T)
    for kind in (score.reasons, score.conditions):
        if kind:
            lo_hi = {m: tuple(round(x, 2) for x in kind.band(m)) for m in ("precision", "recall", "f1")}
            print(f"{kind.kind:10s} similarity of matched pairs: {sorted(kind.primary.similarity_scores, reverse=True)}"
                  f"  | judge band over {SAMPLES} sample(s): {lo_hi}")

## 3. Alignment: reasons

Green = matched well, amber = matched but weak (<0.65: right topic, different substance), red = **missing** (reference reason the system did not produce, hurts recall), blue = **extra** (generated reason with no reference counterpart, hurts precision). `ref_idx` / `gen_idx` are positions in the original lists.

In [ ]:
if score:
    show(alignment_rows(score.reasons).drop(columns=["score"]))

## Alignment: conditions (our extension)

In [ ]:
if score and score.conditions:
    show(alignment_rows(score.conditions).drop(columns=["score"]))
elif score:
    print("No conditions on either side.")

## 4. Failure triage

For every missing, extra or weak reason the table below shows the issues most related to its text (lexical overlap) and what the run did with them:

- strong overlap with an issue that was **not routed** -> routing failure
- routed, but `citations` do not contain the policy you expected -> retrieval
- assessment missed or mis-judged it -> assessment
- assessment found it but the final reason list dropped it -> synthesis
- facts missing the information -> extraction
- everything right but worded differently -> wording / judge-error

These are hints only; look at the evidence, then record **your** diagnosis with `label(...)` below.

In [ ]:
def triage(result, kinds=("missing", "weak", "extra")):
    rows = alignment_rows(result)
    rows = rows[rows["status"].isin(kinds)]
    bundle = run_case.bundle
    for _, row in rows.iterrows():
        text = row["reference"] or row["generated"]
        side = "reference" if row["reference"] else "generated"
        print("=" * 100)
        print(f"[{row['status'].upper()}] ({side} #{row['ref_idx'] if side == 'reference' else row['gen_idx']}) {text}")
        if row["status"] == "weak":
            print(f"   matched to ({row['similarity']}): {row['generated']}")
        if bundle is None:
            print("   (no bundle.json for this run)")
            continue
        for ev in link_evidence(text, bundle):
            routed = f"routed, {ev.finding}{' SUBSTANTIVE' if ev.substantive else ''}, conf={ev.confidence}" if ev.routed else "NOT ROUTED"
            print(f"   issue {ev.issue:32s} overlap={ev.overlap:.2f}  {routed}")
            for c in ev.citations[:3]:
                print(f"        cites: {c}")
        for fact in link_facts(text, bundle):
            print(f"   fact: {fact}")

if score:
    triage(score.reasons)

### Record your diagnosis

`label(kind, index, stage, ...)` stores a label in `outputs/labels/<case>.json`. `index` is the `ref_idx` (for missing/weak reference items) or `gen_idx` with `side="generated"` (for extras). `stage` is one of `extraction, routing, retrieval, assessment, synthesis, wording, judge-error`; `scope` is `general` (would help other cases too) or `case-specific` (be wary: overfitting to five cases).

In [ ]:
def label(kind, index, stage, scope="general", note="", side="reference"):
    result = getattr(score, kind)
    items = result.expected if side == "reference" else result.generated
    status = {"reference": "missing", "generated": "extra"}[side]
    save_label(settings, Label(case=CASE, kind=kind, status=status, item=items[index],
                               stage=stage, scope=scope, note=note))

# Examples (edit and run):
# label("reasons", 2, "routing", note="flood risk never routed")
# label("reasons", 4, "wording", scope="case-specific", side="generated")

display(pd.DataFrame([l.model_dump() for l in load_labels(settings, CASE)]) if load_labels(settings, CASE) else "no labels yet")

## 5. Intermediate data

In [ ]:
bundle = run_case.bundle
assert bundle is not None, "no bundle.json for this run"
print("PROPOSAL:", bundle.facts.proposal)
print("CONSTRAINTS:", bundle.constraints_note)
print("\nFACTS:")
print(bundle.facts.model_dump_json(indent=1))

In [ ]:
display(pd.DataFrame([
    {"issue": a.issue.value, "finding": a.finding.value, "substantive": a.substantive,
     "confidence": a.confidence, "evidence": len(a.evidence), "policy_refs": len(a.policy_refs),
     "conditions": len(a.proposed_conditions), "citations": len(bundle.citations.get(a.issue.value, []))}
    for a in bundle.assessments
]))

Drill into one issue: its rationale, evidence, the policy chunks the assessor saw (from the trace), and what it proposed.

In [ ]:
ISSUE = "residential_amenity" #bundle.assessments[0].issue.value  # change me, e.g. "flood_risk_and_drainage"
a = next(x for x in bundle.assessments if x.issue.value == ISSUE)
print(a.model_dump_json(indent=1))
print("\nCITATIONS:", *bundle.citations.get(ISSUE, []), sep="\n  ")
print("\nPOLICY TEXT THE ASSESSOR SAW:\n" + (policy_seen(run_case, ISSUE) or "(no trace)"))

In [ ]:
display(Markdown(run_case.brief or "_no brief.md_"))

## 6. LLM trace

Every call with stage, model, cache hit, cost. Use `show_call(i)` to read the exact system prompt, user prompt and raw reply for tuning prompts.

In [ ]:
tf = trace_frame(run_case)
display(tf if len(tf) else "no trace for this run")
if len(tf):
    print(tf.groupby("stage")[["cost_usd", "prompt_chars", "reply_chars"]].sum().round(4))

In [ ]:
def show_call(i, max_chars=6000):
    t = run_case.trace[i]
    print(f"#{i} stage={t['stage']} model={t['model']} cached={t['cached']} cost=${t['cost_usd']:.4f} images={t['n_images']}")
    for name in ("system", "user", "reply"):
        print(f"\n----- {name.upper()} -----")
        text = str(t[name])
        print(text[:max_chars] + ("\n...[truncated]" if len(text) > max_chars else ""))

# show_call(0)
# first assess call: show_call(next(i for i, t in enumerate(run_case.trace) if t["stage"].startswith("assess:")))

## 7. Roll-up across the eval cases

Scores every eval case in `RUN` (free if the judge cache is warm) and totals your failure labels. Treat the aggregate cautiously: five cases, four approvals. A fix tagged `case-specific` that lifts one case may not generalise; prefer `general` fixes.

In [ ]:
dev_scores = list((await score_run(judge, settings, RUN, DEV_CASES, SAMPLES)).values())
card = scorecard(dev_scores)
display(card.round(2))
print("mean:\n", card.mean(numeric_only=True).round(3)[["match", "precision", "recall", "f1", "cond_precision", "cond_recall", "cond_f1"]].to_string())
display(label_summary(settings, DEV_CASES))

## 8. Drill into one case: A vs B, reason by reason

Set `COMPARE_RUN` at the top (any saved run, e.g. the `NEW_RUN` from section 9). Reference items are shared, so each row shows how much better/worse run B covered that reference reason (score 0 = missing). For the all-cases view, use section 10.

In [ ]:
COMPARE_RUN = "experiment-20261007-1815-strong" #"experiment-20261007-1615"
CASE = "case-001"

In [ ]:
if COMPARE_RUN and reference:
    b_case = load_run(settings, CASE, COMPARE_RUN)
    score_b = await score_case(judge, CASE, reference, b_case.decision, SAMPLES)
    print(f"A={RUN}  B={COMPARE_RUN}")
    for a_kind, b_kind in ((score.reasons, score_b.reasons), (score.conditions, score_b.conditions)):
        if a_kind and b_kind:
            print(f"\n{a_kind.kind}: F1 {a_kind.primary.f1:.2f} -> {b_kind.primary.f1:.2f}")
            show(compare_kind(a_kind, b_kind), status_col="none")
else:
    print("Set COMPARE_RUN (and use a dev CASE) to compare two runs.")

## 9. Re-run the eval cases into a new named run

Use this after changing the method (prompts, routing, retrieval, models). Results go to `outputs/runs/<NEW_RUN>/` with traces and a `meta.json` recording the models used; the "latest" copies under `outputs/<case>/` are **not** touched, and an existing run is never overwritten unless you set `OVERWRITE`.

Two independent switches control cost:

- `MODELS`: **`"cheap"`** (default) runs every stage on the cheap model; **`"strong"`** uses the configured strong model for vision, assessment and synthesis (about $0.3-0.5 per case); any other string is used as a model id for every stage (e.g. a mid-priced model).
- `CACHE_ONLY`: `True` makes the run free: it uses saved replies only and reports a failure for any case that needs an uncached call. Unchanged prompts are cached, changed ones are not, so this shows how much a change would reuse.

Anything other than `"cheap"` needs `CONFIRM_SPEND = True` (unless `CACHE_ONLY`), so an expensive run is never an accident.

Fair A/B needs the same models on both sides (section 10 warns when they differ). The baseline was made with the strong model, so a `"cheap"` rerun mostly measures cheap-vs-strong, not your change. For a clean comparison of a code change either use `"strong"` once your budget allows, or first make a cheap baseline of the *old* method (check out the old code, run this cell with `NEW_RUN = "baseline-cheap"`) and point section 10's `A_RUN` at it.

In [ ]:
DEV_CASES

In [ ]:
RERUN = True                 # flip to True to execute
NEW_RUN = "experiment-20261011-1913-jev-reasoning-on"      # name for this run under outputs/runs/
MODELS = "strong" #"cheap"              # "cheap" | "strong" | any model id
CACHE_ONLY = False            # True = free replay; uncached cases fail instead of spending
CONFIRM_SPEND = True         # must be True for MODELS != "cheap" when CACHE_ONLY is False
RERUN_CASES = DEV_CASES
OVERWRITE = True

if RERUN:
    if MODELS != "cheap" and not CACHE_ONLY and not CONFIRM_SPEND:
        raise SystemExit(f"MODELS={MODELS!r} can be expensive; set CONFIRM_SPEND = True to proceed.")
    run_settings = settings_for_models(settings, MODELS)
    print(f"MODELS={MODELS!r} CACHE_ONLY={CACHE_ONLY}: cheap={run_settings.cheap_model} | "
          f"strong={run_settings.strong_model} | {len(RERUN_CASES)} cases -> outputs/runs/{NEW_RUN}/")
    report = await rerun_cases(run_settings, RERUN_CASES, NEW_RUN,
                               cache_only=CACHE_ONLY, overwrite=OVERWRITE)
    print(f"saved {report.cases}: {report.api_calls} API calls ({report.cached_calls} served from cache), "
          f"${report.cost_usd:.4f}")
    for case_id, err in report.errors.items():
        print(f"FAILED {case_id}: {err}")
else:
    print("RERUN is False: nothing executed.")

## 10. A/B test: baseline vs new run, across all eval cases

Per case: reasons precision / recall / F1 and conditions F1 for A (baseline) and B (new), their deltas, whether the decision still matches, and a verdict. The `MEAN` row is the headline.

Caveats to keep in view:
- **Same models?** The warnings below say if the two runs used different models; if so the comparison is confounded.
- **Judge noise.** The judge is cached per (reference, generated) text pair, so an *unchanged* output scores identically on both sides, but a *changed* output gets a fresh, noisy judgement. With `SAMPLES = 1` a small delta can be noise; set `SAMPLES = 3` (a few more cheap judge calls) and the verdict shows `within noise` when the delta is smaller than the judge's own spread.
- **Five cases, four approvals.** A gain on one case may not generalise; check the per-reason view in section 8 before believing it.

In [ ]:
# NEW_RUN = "routing-llm"
NEW_RUN = "experiment-20261007-1918-strong"

In [ ]:
settings = settings.all_cheap()
settings

In [ ]:
A_RUN = "experiment-20261007-1918-strong"           # "@baseline" = the current baseline pointer
B_RUN = NEW_RUN               # the run to test

a_run = BASELINE if A_RUN == "@baseline" else A_RUN
for w in model_mismatch(settings, a_run, B_RUN):
    print("WARNING (confounded A/B):", w)
display(pd.DataFrame({"A": load_meta(settings, a_run), "B": load_meta(settings, B_RUN)}).drop(["report"], errors="ignore"))

scores_a = await score_run(judge, settings, a_run, DEV_CASES, SAMPLES)
scores_b = await score_run(judge, settings, B_RUN, DEV_CASES, SAMPLES)
print(f"A={a_run} ({len(scores_a)} cases)   B={B_RUN} ({len(scores_b)} cases)")
if scores_a and scores_b:
    ab = ab_table(scores_a, scores_b)
    cols = ["f1_a", "f1_b", "f1_delta", "precision_delta", "recall_delta", "cond_f1_delta", "match_a", "match_b", "verdict"]
    display(ab[cols].round(3))
else:
    print("Need saved runs for both A and B (see section 9).")

## 11. Optionally promote the new run to be the baseline

Moves the baseline **pointer** (`outputs/runs/_baseline`); no run is copied or deleted, so the old baseline stays available and you can point back to it. Re-run the first code cell afterwards so `BASELINE` / `RUN` pick it up. Only the eval cases need to be present in the new run.

In [ ]:
PROMOTE = False

if PROMOTE:
    previous = set_baseline(settings, NEW_RUN)
    print(f"baseline: {previous} -> {NEW_RUN}   (undo with set_baseline(settings, {previous!r}))")
else:
    print(f"PROMOTE is False: baseline stays {BASELINE!r}.")